In [1]:

import os
import json
import random
from tqdm import tqdm
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForCausalLM,
    GenerationConfig,
)
import torch
import torch.nn.functional as F
from fancy_einsum import einsum
import einops
import plotly.graph_objs as go
from plotly.subplots import make_subplots

from src.record_utils import record_activations, get_module, untuple_tensor
from src.utils import load_model
from src.HookedQwen import convert_to_hooked_model
from IPython.core.debugger import Pdb

In [2]:

cos = F.cosine_similarity

In [3]:

base_dir = "/n/home01/ajyl/verify_circuit"

In [4]:


def seed_all(seed, deterministic_algos=False):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)
    if deterministic_algos:
        torch.use_deterministic_algorithms()


def unembed(vector, lm_head, k=10):
    dots = einsum("vocab d_model, d_model -> vocab", lm_head, vector.to(lm_head.device))
    top_k = dots.topk(k).indices
    return top_k


def unembed_text(vector, lm_head, tokenizer, k=10):
    top_k = unembed(vector, lm_head, k=k)
    return tokenizer.batch_decode(top_k, skip_special_tokens=True)

In [5]:


def _add_o_proj_hook(model, layer_idx, head_idx):
    def hook(module, input, output):
        # output.shape: [batch, heads, seq, head_dim]
        output[:, :, head_idx, :] = 0
        return output

    module = model.model.layers[layer_idx].self_attn.hook_attn_out_per_head
    return module.register_forward_hook(hook)

In [6]:


def _turn_off_mlp(model, layer_idx, mlp_idxs):
    def hook(module, input, output):
        output[:, -1, mlp_idxs] = 0
        return output

    module = model.model.layers[layer_idx].mlp.hook_mlp_mid
    return module.register_forward_hook(hook)

In [7]:


def add_hooks(model, hook_config):
    handles = []
    for hook_module, layer, head_idx in hook_config:
        if hook_module == "attn_out":
            hook_func = _add_o_proj_hook
        elif hook_module == "mlp":
            hook_func = _turn_off_mlp
        handles.append(hook_func(model, layer, head_idx))
    return handles

In [8]:


@torch.no_grad()
def generate_hooked(
    model,
    input_ids,
    attention_mask,
    max_new_tokens,
    block_size,
    tokenizer,
    hook_config,
):
    """
    Generate text using a transformer language model with greedy sampling.

    Args:
        model: The auto-regressive transformer model that outputs logits.
        input_ids: A tensor of shape (batch_size, sequence_length) representing the initial token indices.
        max_new_tokens: The number of new tokens to generate.
        block_size: The maximum sequence length (context window) the model can handle.
        device: The device on which computations are performed.

    Returns:
        A tensor containing the original context concatenated with the generated tokens.
    """
    remove_all_hooks(model)
    device = model.device
    model.eval()  # Set the model to evaluation mode
    eos_token_id = tokenizer.eos_token_id

    input_ids = input_ids.clone()
    attention_mask = attention_mask
    batch_size = input_ids.shape[0]

    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    token_open = tokenizer.encode(" (")[0]  # 320

    for _ in range(max_new_tokens):
        if finished.all():
            break

        if input_ids.shape[1] > block_size:
            idx_cond = input_ids[:, -block_size:]
            attn_mask_cond = attention_mask[:, -block_size:]
        else:
            idx_cond = input_ids
            attn_mask_cond = attention_mask

        position_ids = attn_mask_cond.long().cumsum(-1) - 1
        position_ids.masked_fill_(attn_mask_cond == 0, 1)

        output = model(
            idx_cond,
            attention_mask=attn_mask_cond,
            position_ids=position_ids,
            return_dict=True,
        )
        logits = output["logits"]
        logits = logits[:, -1, :]  # shape: (batch, vocab_size)
        next_token = torch.argmax(logits, dim=-1, keepdim=True)  # shape: (batch, 1)

        most_recent_token = [
            tokenizer.decode(idx_cond[batch_idx, -1]) for batch_idx in range(batch_size)
        ]

        interv_batch_idx = []
        for batch_idx in range(batch_size):
            if most_recent_token[batch_idx] == " (":
                interv_batch_idx.append(batch_idx)

        if len(interv_batch_idx) > 0:

            handles = add_hooks(model, hook_config)
            interv_output = model(
                idx_cond[interv_batch_idx],
                attention_mask=attn_mask_cond[interv_batch_idx],
                position_ids=position_ids[interv_batch_idx],
                return_dict=True,
            )
            logits = interv_output["logits"]
            logits = logits[:, -1, :]  # shape: (batch, vocab_size)
            interv_next_token = torch.argmax(logits, dim=-1, keepdim=True)
            next_token[interv_batch_idx] = interv_next_token

            for handle in handles:
                handle.remove()

        new_finished = (~finished) & (next_token.squeeze(1) == eos_token_id)
        finished |= new_finished
        next_token[finished] = eos_token_id

        # Append the predicted token to the sequence
        input_ids = torch.cat([input_ids, next_token], dim=1)
        new_mask = torch.ones(
            (batch_size, 1), dtype=attention_mask.dtype, device=device
        )
        attention_mask = torch.cat([attention_mask, new_mask], dim=1)

    return input_ids

In [9]:


def get_mlp_value_vecs(model):
    mlp_value_vecs = [layer.mlp.down_proj.weight.cpu() for layer in model.model.layers]
    # [n_layers, d_mlp (11008), d_model (2048)]
    return torch.stack(mlp_value_vecs, dim=0)

In [10]:


def remove_all_hooks(model):
    for (
        name,
        module,
    ) in model.named_modules():  # Recursively iterates through submodules
        if hasattr(module, "_forward_hooks"):
            for handle_id in list(module._forward_hooks.keys()):
                module._forward_hooks.pop(handle_id)

In [11]:

config = {
    "model_path": os.path.join(
        base_dir, "checkpoints/TinyZero/v4/actor/global_step_300"
    ),
    "probe_path": os.path.join(base_dir, "probe_checkpoints/v2/probe.pt"),
    "batch_size": 1,
    "max_prompt_length": 256,
    "max_response_length": 300,
    "n_layers": 36,
    "d_model": 2048,
    "seed": 42,
}

In [12]:

seed_all(config["seed"])
assert torch.cuda.is_available()

model_path = config["model_path"]
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-14B")

In [13]:

qwen = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
    trust_remote_code=True,
    device_map="auto",
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [14]:

convert_to_hooked_model(qwen)

In [15]:

generation_config = GenerationConfig(do_sample=False)

In [16]:

token_this = tokenizer.encode("this")[0]  # 574
token_equals = tokenizer.encode("equals")[0]
token_open = tokenizer.encode(" (")[0]  # 320
token_not = tokenizer.encode("not")[0]  # 1921

In [17]:

samples = torch.load(os.path.join(base_dir, "data/countdown/icl_base_run.pt"))

/tmp/ipykernel_2003231/1117546413.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  samples = torch.load(os.path.join(base_dir, "data/countdown/icl_base_run.pt"))


In [18]:

probe_model = torch.load(config["probe_path"]).detach().cuda()

/tmp/ipykernel_2003231/2924032859.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  probe_model = torch.load(config["probe_path"]).detach().cuda()


In [65]:


def run(actor, samples, hook_config, batch_size):
    max_gen_length = 100

    this_timesteps = []
    all_generations = []
    odd_batches = []

    total = 0
    success = 0
    mix = 0
    fail = 0

    test_size = len(samples)
    for batch_idx in tqdm(range(0, test_size, batch_size)):
        curr_batch = samples[batch_idx : batch_idx + batch_size]
        
        _this_timestep = [sample["this_timestep"] - 1 for sample in curr_batch]
        _input_ids = [
            curr_batch[_idx]["response"][: _this_timestep[_idx]]
            for _idx in range(len(curr_batch))
        ]
        max_length = max(seq.shape[0] for seq in _input_ids)
        padded_input_ids = []
        for seq in _input_ids:
            pad_length = max_length - seq.shape[0]
            padded = F.pad(seq, (pad_length, 0), value=tokenizer.pad_token_id)
            padded_input_ids.append(padded)
        input_ids = torch.stack(padded_input_ids, dim=0)
        attention_mask = input_ids != tokenizer.pad_token_id

        hooked_output = generate_hooked(
            actor,
            input_ids,
            attention_mask,
            max_gen_length,
            2000,
            tokenizer,
            hook_config,
        )
        hooked_output_text = tokenizer.batch_decode(
            hooked_output, skip_special_tokens=True
        )
        all_generations.append(hooked_output_text)

        generated_ids = hooked_output[:, input_ids.shape[1] :]
        generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        for generation in generated_text:
            if "this works" not in generation and "<answer>" not in generation:
                success += 1
            elif "this works" not in generation and "<answer>" in generation:
                mix += 1
            elif "this works" in generation and "<answer>" in generation:
                fail += 1
            else:
                print("HMM??")
                print("Likely didn't reach <answer> token within 100 tokens.")
                print(generation)

        total += len(curr_batch)

    return success / total, mix / total, fail / total

In [20]:


def build_mlp_hook_config(actor, probe_model, layers, k):

    labels = [0, 1]
    value_vecs = get_mlp_value_vecs(actor)
    hook_config = []
    top_cos_scores = {label: [] for label in labels}
    for label in labels:
        for target_probe_layer in range(18, 36):
            target_probe = probe_model[target_probe_layer, :, label]
            model_layer = int(target_probe_layer / 36 * 48)

            for layer_idx in range(0, model_layer + 1):
                cos_scores = cos(
                    value_vecs[layer_idx].to(target_probe.device),
                    target_probe.unsqueeze(-1),
                    dim=0,
                )
                _topk = cos_scores.topk(k=k)
                _values = [x.item() for x in _topk.values]
                _idxs = [x.item() for x in _topk.indices]
                topk = list(
                    zip(
                        _values,
                        _idxs,
                        [target_probe_layer] * _topk.indices.shape[0],
                        [layer_idx] * _topk.indices.shape[0],
                    )
                )
                top_cos_scores[label].extend(topk)

    _sorted_scores_0 = sorted(top_cos_scores[0], key=lambda x: x[0], reverse=True)
    _sorted_scores_1 = sorted(top_cos_scores[1], key=lambda x: x[0], reverse=True)

    _unique = set()
    sorted_scores_0 = []
    for entry in _sorted_scores_0:
        _pair = (entry[3], entry[1])
        if _pair not in _unique:
            _unique.add(_pair)
            sorted_scores_0.append(("mlp", _pair[0], _pair[1]))

    _unique = set()
    sorted_scores_1 = []
    for entry in _sorted_scores_1:
        _pair = (entry[3], entry[1])
        if _pair not in _unique:
            _unique.add(_pair)
            sorted_scores_1.append(("mlp", _pair[0], _pair[1]))

    return sorted_scores_0, sorted_scores_1

In [49]:


def _get_occurrence_idxs(hay, needle):
    window_size = needle.shape[0]
    hay = hay.unfold(0, window_size, 1)
    mask = (hay == needle).all(dim=1)
    offset = window_size - 1
    match_idxs = mask.nonzero(as_tuple=True)[0] + offset
    return match_idxs


@torch.no_grad()
def _get_attn_density_for_target(actor, samples, batch_size):

    device = actor.device
    n_layers = 48
    record_module_names = [
        f"model.layers.{idx}.self_attn.hook_attn_pattern" for idx in range(n_layers)
    ]
    test_size = len(samples)
    _all_attn_pattern = []
    all_recording = {}
    cutoff = (
        tokenizer(" Let's try different", return_tensors="pt")["input_ids"]
        .squeeze()
        .to(device)
    )
    all_attn_density = []
    for batch_idx in tqdm(range(0, test_size, batch_size)):
        curr_batch = samples[batch_idx : batch_idx + batch_size]
        _this_timestep = [sample["this_timestep"] for sample in curr_batch]

        _input_ids = [
            curr_batch[_idx]["response"][: _this_timestep[_idx]]
            for _idx in range(len(curr_batch))
        ]
        max_length = max(seq.shape[0] for seq in _input_ids)
        padded_input_ids = []
        for seq in _input_ids:
            pad_length = max_length - seq.shape[0]
            padded = F.pad(seq, (pad_length, 0), value=tokenizer.pad_token_id)
            padded_input_ids.append(padded)
        input_ids = torch.stack(padded_input_ids, dim=0)
        attention_mask = input_ids != tokenizer.pad_token_id
        position_ids = attention_mask.long().cumsum(-1) - 1
        position_ids.masked_fill_(attention_mask == 0, 1)

        with record_activations(actor, record_module_names) as recording:
            output = actor(
                input_ids,
                attention_mask=attention_mask,
                position_ids=position_ids,
                return_dict=True,
            )

        # [layers, batch, heads, seq]
        _attn_pattern = torch.stack(
            [
                recording[f"model.layers.{layer_idx}.self_attn.hook_attn_pattern"][0][
                    :, :, -1, :
                ].to("cuda:1")
                for layer_idx in range(n_layers)
            ]
        )
        attn_density = []
        for _idx in range(len(curr_batch)):
            target_tokens = tokenizer(
                str(curr_batch[_idx]["target"]), return_tensors="pt"
            )["input_ids"].squeeze()
            if target_tokens[0] == 151646: # (BOS):
                target_tokens = target_tokens[1:]

            _context = input_ids[_idx]
            cutoff_idx = _get_occurrence_idxs(_context, cutoff)
            #curr_context = _context[: cutoff_idx[0]]
            match_idxs = _get_occurrence_idxs(
                _context, target_tokens.to(_context.device)
            ).to(_attn_pattern.device)

            # [layers, heads]
            density = _attn_pattern[:, _idx, :, match_idxs].sum(dim=-1)
            attn_density.append(density)

    all_attn_density = torch.stack(attn_density, dim=0)
    return all_attn_density.mean(dim=0)


def _get_prev_token_heads(actor, samples, batch_size, thresh=0.1):

    attn_pattern = _get_attn_density_for_target(actor, samples, config["batch_size"])
    top_values, top_idxs = torch.topk(attn_pattern.flatten(), 1000)
    top_idxs = np.array(np.unravel_index(top_idxs.cpu().numpy(), attn_pattern.shape)).T
    prev_token_heads = top_idxs[
        (top_values > thresh).nonzero().squeeze().cpu()
    ].squeeze()
    return prev_token_heads, attn_pattern



In [34]:


def get_WO_WV_OV(actor):

    n_layers = 48
    n_heads = actor.config.num_attention_heads
    n_kv_heads = actor.config.num_key_value_heads
    n_kv_groups = n_heads // n_kv_heads
    W_O = []
    W_V = []
    for idx in range(n_layers):

        _W_O = actor.model.layers[idx].self_attn.o_proj.weight
        _W_O = einops.rearrange(_W_O, "m (n h)->n h m", n=n_heads)
        W_O.append(_W_O.cpu())

        _W_V = actor.model.layers[idx].self_attn.v_proj.weight
        _W_V = einops.rearrange(_W_V, "(n h) m->n m h", n=n_kv_heads)
        _W_V = torch.repeat_interleave(_W_V, dim=0, repeats=n_kv_groups)
        W_V.append(_W_V.cpu())

    # [layers, heads, d_head, d_model]
    W_O = torch.stack(W_O, dim=0)
    W_V = torch.stack(W_V, dim=0)
    OV = einsum(
        "layers heads d_head d_model, layers heads d_model d_head -> layers heads d_model",
        W_O,
        W_V,
    )
    return W_O, W_V, OV


def get_OV_for_attn_heads(actor, OV, attn_heads):
    OVs = []
    for attn_head in attn_heads:
        layer_idx = attn_head[0]
        head_idx = attn_head[1]
        OVs.append(OV[layer_idx, head_idx].cpu())
    return torch.stack(OVs, dim=0)


def get_verification_heads(
    actor, samples, prev_token_heads, probe_model, num_mlp_vecs=200
):
    _, top_scores_1 = build_mlp_hook_config(actor, probe_model, list(range(24, 48)), 50)
    top_scores_1 = [(x[1], x[2]) for x in top_scores_1]

    gate_vecs = torch.stack(
        [
            actor.model.layers[x[0]].mlp.gate_proj.weight[x[1]].cpu()
            for x in top_scores_1[:num_mlp_vecs]
        ],
        dim=0,
    )
    up_proj_vecs = torch.stack(
        [
            actor.model.layers[x[0]].mlp.up_proj.weight[x[1]].cpu()
            for x in top_scores_1[:num_mlp_vecs]
        ],
        dim=0,
    )

    W_O, W_V, _OV = get_WO_WV_OV(actor)
    OV = get_OV_for_attn_heads(actor, _OV, prev_token_heads)

    gate_vecs = gate_vecs.to("cuda:1")
    up_proj_vecs = up_proj_vecs.to("cuda:1")
    OV = OV.to("cuda:1")

    dots_gate = einsum("N d_model, L d_model -> N L", gate_vecs, OV)
    act_fn = actor.model.layers[0].mlp.act_fn
    acts = act_fn(dots_gate)

    dots_up_proj = einsum("N d_model, L d_model -> N L", up_proj_vecs, OV)
    weights = (acts * dots_up_proj).mean(dim=0)
    top_val, top_idx = torch.topk(weights.flatten(), k=len(prev_token_heads))
    top_idx = np.array(
        np.unravel_index(top_idx.cpu().numpy(), weights.shape)
    ).T.squeeze()
    verif_heads = [prev_token_heads[x].tolist() for x in top_idx]
    return verif_heads


def build_attn_hook_config(
    actor,
    samples,
    batch_size,
    probe_model,
    prev_token_thresh=0.1,
    num_mlp_vecs=200,
):
    prev_token_heads, attn_pattern = _get_prev_token_heads(
        actor, samples, batch_size, prev_token_thresh
    )
    print(prev_token_heads)
    verif_heads = get_verification_heads(
        actor, samples, prev_token_heads, probe_model, num_mlp_vecs
    )
    heads = [("attn_out", layer, head_idx) for layer, head_idx in verif_heads]
    return heads, attn_pattern

In [23]:

batch_size = config["batch_size"]
dev_size = 50

In [24]:

# Orig:
# hook_config = []
# orig_success, orig_mix, orig_fail = run(
#   actor,
#   samples[:dev_size],
#   hook_config,
#   batch_size,
# )
# print(f"Orig success: {orig_success}")
# print(f"Orig mix: {orig_mix}")
# print(f"Orig fail: {orig_fail}")

In [25]:


def get_transfer_matrix(device):
    """
    Get transfer matrix between model `src` and model `dst`.
    """
    src_token_embeds = torch.load(os.path.join(base_dir, "checkpoints/qwen2.5_3B_lm_head.pt"))
    num_samples = 100000


    # Grab 100000 random indices from the tokenizer
    random_tokens = torch.randint(0, src_token_embeds.shape[0], (num_samples,))
    # [100, 1024]
    src_tokens = src_token_embeds[random_tokens]

    #dst_token_embeds = dst.lm_head.weight.detach()
    dst_token_embeds = torch.load(os.path.join(base_dir, "checkpoints/qwen2.5_14B_lm_head.pt"))
    # [100, 1024]
    dst_tokens = dst_token_embeds[random_tokens]

    # Random initialization
    A = src_tokens.to(device)
    B = dst_tokens.to(device)

    # Ensure that A and B have the same number of rows
    assert A.size(0) == B.size(0), "A and B must have the same number of rows."

    # Ridge parameter (regularization strength)
    alpha = 0.1

    # Setup the problem in terms of least squares with regularization
    A_transpose_A = A.t() @ A
    I = torch.eye(A.shape[1]).to(A_transpose_A.device)

    # Regularized least squares
    reg_matrix = A_transpose_A + alpha * I
    A_transpose_B = A.t() @ B

    # Compute T with regularization
    T = torch.linalg.solve(reg_matrix, A_transpose_B)

    # To verify the transformation
    B_transformed = A @ T
    error = torch.norm(B - B_transformed)
    return T, error


def get_transfer_probe(orig_probe):
    device = orig_probe.device
    transfer_matrix, error = get_transfer_matrix(device)
    return einsum(
        "layer d_model class, d_model d_model_hat -> layer d_model_hat class",
        orig_probe.to(device),
        transfer_matrix.to(device),
    )

In [26]:

dev_size = 10

In [27]:

# Orig:
#dev_size = 10
#hook_config = []
#orig_success, orig_mix, orig_fail = run(
#   qwen,
#   samples[:dev_size],
#   hook_config,
#   batch_size,
#)
#print(f"Orig success: {orig_success}")
#print(f"Orig mix: {orig_mix}")
#print(f"Orig fail: {orig_fail}")

In [28]:

# [Layer, d_model (Qwen), 2]
new_probe = get_transfer_probe(probe_model)

/tmp/ipykernel_2003231/612592750.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  src_token_embeds = torch.load(os.path.join(base_dir, "checkpoints/qwen2.5_3B_lm_head.pt"

In [29]:

# MLP (only [1]):

print("Running MLP (only [1])")
mlp_hook_config_0, mlp_hook_config_1 = build_mlp_hook_config(qwen, new_probe, list(range(24, 48)), 50)
val_vecs = get_mlp_value_vecs(qwen)

Running MLP (only [1])


In [30]:

for entry in mlp_hook_config_1[:50]:
    print(
        unembed_text(
            val_vecs[entry[1], :, entry[2]], qwen.lm_head.weight, tokenizer, k=10
        )
    )

['不失', 'NotNull', '得起', '得住', '不惜', '不乏', '值得一', '是可以', '总算', '只需要']
['不失', 'etch', 'NotNull', '性强', '涯', '又能', '更强', 'etypes', 'ETCH', '目前已']
['从来', ' Dudley', 'baugh', 'NullOrEmpty', 'rier', 'erty', ' blindly', 'iker', '暂', '렁']
['而是', '活', '呷', 'stitución', 'Nor', 'uges', ' nor', '谁知道', 'るように', '畀']
['abis', '愎', '逍', '欢喜', 'Fetching', 'fore', ' forgotten', '迩', '照样', 'vik']
['nil', 'ilog', ',nil', 'Nil', '-nil', '适度', 'ところで', 'iminal', 'uit', 'ihilation']
['谁知道', 'isseur', '当之', ' Impl', 'crew', ' Crew', '_builtin', '倏', 'LayoutParams', ':YES']
[' dah', '洙', '公然', '忽然', '<small', '平淡', '_COMPARE', 'libc', 'рев', '具有一定']
['itch', '就必须', '呗', '必然会', 'Und', '骡', 'erot', '分明', '就会', 'NotNull']
['删除成功', ' successes', 'Success', ' success', ' succeeded', ' favorable', 'UCCESS', '.Success', '成功的', 'Successful']
[' atol', '明知', '玲', ' Saunders', 'sei', '淡化', ' Torrent', 'iled', 'vac', '祥']
['쇠', '전문', '榻', '挺好', '遗憾', 'yang', '-ignore', ' sondern', '也只能', '_sector']
['也不可能', '也不会', '也没什么',

In [31]:

#mlp_1_success, mlp_1_mix, mlp_1_fail = run(
#    qwen,
#    samples[:dev_size],
#    mlp_hook_config_1[:200],
#    batch_size,
#)
#print(f"MLP 1 success: {mlp_1_success}")
#print(f"MLP 1 mix: {mlp_1_mix}")
#print(f"MLP 1 fail: {mlp_1_fail}")

In [32]:

# MLP (Both [0, 1]):

#print("Running MLP (both [0, 1])")
#mlp_both_hook_config = mlp_hook_config_0 + mlp_hook_config_1
#mlp_both_success, mlp_both_mix, mlp_both_fail = run(
#    qwen,
#    samples[:dev_size],
#    mlp_both_hook_config,
#    batch_size,
#)
#print(f"MLP both success: {mlp_both_success}")
#print(f"MLP both mix: {mlp_both_mix}")
#print(f"MLP both fail: {mlp_both_fail}")

In [50]:

# Attention:

print("Running Attention")
hook_config, attn_pattern = build_attn_hook_config(
    qwen,
    samples[:dev_size],
    batch_size,
    new_probe,
    prev_token_thresh=0.1,
    num_mlp_vecs=200,
)
print(hook_config)
print(len(hook_config))

Running Attention


100%|████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:11<00:00,  1.12s/it]


[[46 16]
 [35  3]
 [ 1 28]
 [33  9]
 [18  2]
 [ 2 19]
 [39 21]
 [ 5 11]
 [11  8]
 [40 19]
 [24  3]
 [33  5]
 [26 39]
 [43  2]
 [ 2 13]
 [16 27]
 [16 25]
 [27  5]
 [30  4]
 [20  2]
 [27 16]
 [43  4]
 [32 20]
 [24 19]
 [40 16]
 [46 19]
 [27 15]
 [23 14]
 [41 36]
 [24  6]
 [16 11]
 [39 23]
 [23 16]
 [44  0]
 [35 33]
 [ 5 12]
 [28 18]
 [31 19]
 [39 19]
 [31 10]
 [30 21]
 [28 14]
 [32  0]
 [22 25]
 [43  1]
 [46 18]
 [31 39]
 [35  4]
 [31 33]
 [ 2 14]
 [30 11]
 [34 30]
 [39 27]
 [30  8]
 [37 33]
 [38 11]
 [43  0]
 [ 4  9]
 [28 29]
 [27 31]
 [13  2]
 [44  4]
 [31  7]
 [28 27]
 [33  3]
 [23 11]
 [ 2 16]
 [47 39]
 [23 12]
 [ 3 13]
 [30 32]
 [27 12]
 [ 1  6]
 [32 26]
 [15 14]
 [30  1]
 [47 30]
 [31 24]
 [21 32]
 [24  0]
 [29 17]
 [37  5]
 [42 14]
 [45 14]
 [28  8]
 [17 20]
 [30 19]
 [31 23]
 [38 16]
 [41 37]
 [26 35]
 [30 27]
 [41  6]
 [38 19]
 [36  1]
 [37  9]
 [12 14]
 [33 30]
 [ 6 37]
 [ 5 13]
 [31 21]
 [27 18]
 [28 35]
 [27 17]
 [39  8]
 [42 11]
 [35 32]
 [30 29]
 [30 18]
 [31  6]
 [30  2]
 

In [66]:

#
_hook_config = hook_config[:156]
#
verif_attn_success, verif_attn_mix, verif_attn_fail = run(
    qwen,
    samples,
    _hook_config,
    batch_size,
)
print(f"Attn verif success: {verif_attn_success}")
print(f"Attn verif mix: {verif_attn_mix}")
print(f"Attn verif fail: {verif_attn_fail}")

  2%|█▍                                                                               | 4/222 [08:43<7:55:15, 130.80s/it]


KeyboardInterrupt: 